# <font color="steelblue">Estado fetal por cardiotocografía</font>

**Material desarrollado por los [equipos de trabajo de IA4LEGOS](https://ia4legos.umh.es/)**

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a>

No olvides hacer una copia si deseas utilizarlo.


## <font color="steelblue">Objetivos del proyecto</font>

A partir de **21 características** extraídas automáticamente de cardiotocogramas (proyecto SisPorto), construir, comparar y **desplegar** un clasificador del **estado fetal**. Lo que distingue a este proyecto es que el dataset ofrece **dos objetivos** y debéis **razonar la formulación**:

* **`NSP`** — estado fetal en **3 clases**: Normal · Sospechoso · **Patológico**. Es la tarea **clínicamente relevante**, **desequilibrada**, con una **clase minoritaria crítica** (Patológico) que **no se puede pasar por alto**.
* **`CLASS`** — patrón morfológico en **10 clases** (A–K). Problema de grano fino.
* Ambos están **jerárquicamente relacionados**: los 10 patrones de `CLASS` agregan en los 3 estados de `NSP`.

El proyecto se centra en **`NSP`** (con la prioridad puesta en el *recall* de **Patológico**), y deja como extensión el problema de **10 clases** y el enfoque **jerárquico**. Aplicaréis además el flujo completo: comparación de modelos, equilibrado **multiclase**, optimización, combinación, interpretación y despliegue.


## <font color="steelblue">El conjunto de datos</font>

### <font color="steelblue">Origen y estructura</font>

El conjunto procede del repositorio UCI y se basa en el trabajo de Ayres-de-Campos y Bernardes (2000), autores del sistema **SisPorto** de análisis automatizado de cardiotocogramas. Contiene **2.126 registros**, cada uno correspondiente a un cardiotocograma fetal (CTG). Los trazados fueron **procesados automáticamente** por SisPorto, que extrajo de cada uno un conjunto de características diagnósticas, y fueron después **clasificados por tres obstetras expertos**, cuyo dictamen de consenso proporciona las etiquetas. Cada CTG recibió dos etiquetas distintas —un **patrón morfológico** y un **estado fetal**—, de modo que el mismo conjunto admite un problema de **10 clases** o uno de **3 clases**.

Conviene entender qué mide un cardiotocograma antes de leer las variables. El aparato registra simultáneamente dos señales: la **frecuencia cardíaca fetal** (FHR, *fetal heart rate*) y la **actividad contráctil del útero** (UC, *uterine contractions*). De la primera se extraen tres familias de descriptores: los **eventos** (aceleraciones, deceleraciones), la **variabilidad** del ritmo latido a latido, y la forma del **histograma** de frecuencias observadas durante el registro. Todas las variables son **numéricas** y **no hay valores faltantes**.

### <font color="steelblue">Diccionario de variables</font>

**Bloque 1 — Frecuencia cardíaca fetal y contracciones uterinas** *(eventos por segundo, salvo `LB`)*

| Variable | Descripción |
|---|---|
| `LB` | **Línea de base** de la frecuencia cardíaca fetal, en latidos por minuto. Es el nivel medio en torno al cual oscila el ritmo. Los valores normales se sitúan en torno a 110–160 lpm; por debajo se habla de bradicardia y por encima, de taquicardia. |
| `AC` | Número de **aceleraciones** por segundo. Son subidas transitorias de la FHR; su presencia es un **signo tranquilizador** de bienestar fetal. |
| `FM` | Número de **movimientos fetales** por segundo detectados durante el registro. |
| `UC` | Número de **contracciones uterinas** por segundo. Es la señal de la actividad del parto y el contexto en que deben leerse las deceleraciones. |
| `DL` | Número de **deceleraciones leves** por segundo (caídas transitorias de la FHR de escasa magnitud). |
| `DS` | Número de **deceleraciones severas** por segundo. Suceso raro; en la práctica esta variable es casi siempre nula. |
| `DP` | Número de **deceleraciones prolongadas** por segundo. Junto con `DS`, son los eventos de mayor gravedad clínica. |

**Bloque 2 — Variabilidad de la frecuencia cardíaca**

La variabilidad —las pequeñas fluctuaciones del ritmo— es el indicador más fino de la integridad del sistema nervioso autónomo fetal: su **pérdida** es un signo de alarma.

| Variable | Descripción |
|---|---|
| `ASTV` | **Porcentaje de tiempo** del registro con variabilidad **anormal a corto plazo** (*short term variability*). Valores altos indican una fracción grande del trazado con variabilidad deficiente. |
| `MSTV` | **Valor medio** de la variabilidad a corto plazo. |
| `ALTV` | **Porcentaje de tiempo** con variabilidad **anormal a largo plazo** (*long term variability*). |
| `MLTV` | **Valor medio** de la variabilidad a largo plazo. |

> **Atención al sentido de las escalas:** `ASTV` y `ALTV` miden *anormalidad* (más es peor), mientras que `MSTV` y `MLTV` miden *variabilidad* (menos es peor). Los cuatro no apuntan en la misma dirección, un detalle esencial al interpretar los coeficientes o los valores SHAP.

**Bloque 3 — Histograma de la frecuencia cardíaca fetal**

Estas diez variables no describen eventos, sino la **distribución** de los valores de FHR observados a lo largo del registro. Son, en esencia, estadísticos descriptivos de esa distribución.

| Variable | Descripción |
|---|---|
| `Width` | **Amplitud** del histograma (máximo menos mínimo). Mide la dispersión global del ritmo. |
| `Min` | Valor **mínimo** del histograma de FHR. |
| `Max` | Valor **máximo** del histograma de FHR. |
| `Nmax` | Número de **picos** (modas locales) del histograma. |
| `Nzeros` | Número de **ceros** del histograma (intervalos de frecuencia sin ninguna observación). |
| `Mode` | **Moda** del histograma: el valor de FHR más frecuente. |
| `Mean` | **Media** del histograma. |
| `Median` | **Mediana** del histograma. |
| `Variance` | **Varianza** del histograma. Junto con `Width`, cuantifica la dispersión. |
| `Tendency` | **Tendencia** o asimetría del histograma, codificada habitualmente como −1 (asimetría izquierda), 0 (simétrico) y +1 (asimetría derecha). Es la única variable de este bloque que no es una medida continua. |

**Bloque 4 — Las dos variables objetivo**

| Variable | Tipo | Valores | Descripción |
|---|---|---|---|
| `NSP` | Categórica **ordinal** (3 clases) | 1 = Normal · 2 = Sospechoso · 3 = Patológico | **Estado fetal** de consenso. Es el objetivo habitual. La asignación sigue una regla jerárquica: *normal* cuando todos los criterios de evaluación lo son; *sospechoso* cuando uno lo es y el resto son normales; y *patológico* cuando al menos un criterio es patológico o dos o más son sospechosos. |
| `CLASS` | Categórica **nominal** (10 clases) | 1–10 | **Patrón morfológico** del trazado. Los diez patrones son: **A** sueño tranquilo · **B** sueño REM · **C** vigilia tranquila · **D** vigilia activa · **E/SH** patrón de cambio (*shift*) · **AD** patrón acelerativo/decelerativo (situación de estrés) · **DE** patrón decelerativo (estimulación vagal) · **LD** patrón ampliamente decelerativo · **FS** patrón plano-sinusoidal (patológico) · **SUSP** patrón sospechoso. |

**Distribución de `NSP`.** El reparto está **muy desequilibrado**: unos 1.655 registros normales (≈78 %), 295 sospechosos (≈14 %) y solo 176 patológicos (≈8 %). Este desequilibrio, unido a que la clase minoritaria es precisamente **la clínicamente crítica**, condiciona por completo la evaluación del modelo.

> **Sobre el código de carga (importante):** el fichero original contiene 40 columnas, de las que solo unas pocas son características utilizables. El código elimina:
> * las columnas de **metadatos** (`FileName`, `Date`, `SegFile`, `b`, `e`), que identifican el fichero y el segmento del trazado y carecen de valor predictivo;
> * una **línea de base redundante** (`LBE`), calculada por un procedimiento alternativo y prácticamente idéntica a `LB`;
> * y, sobre todo, las columnas **`A`, `B`, …, `SUSP`**, que son **indicadores *one-hot* de los propios objetivos**: cada una vale 1 si el registro pertenece a ese patrón morfológico. Utilizarlas como predictoras sería una **fuga de información** flagrante —equivaldría a darle al modelo la respuesta—, y explicaría una exactitud sospechosamente perfecta.
>
> Tras ese filtrado quedan las **21 características** + `CLASS` + `NSP`.
>
> **Ojo:** las variables del **histograma** (`Mode`, `Mean`, `Median`, `Min`, `Max`, `Width`…) están **muy correlacionadas** entre sí (multicolinealidad), algo esperable: son estadísticos calculados sobre la misma distribución. `Width` es, de hecho, una función exacta de `Max` y `Min`.

### <font color="steelblue">Advertencias metodológicas</font>

1. **Dos objetivos, nunca a la vez.** `CLASS` y `NSP` etiquetan el mismo registro desde dos perspectivas y están fuertemente relacionadas (los patrones `FS` y `SUSP`, por ejemplo, anticipan un estado no normal). Emplear una como predictora de la otra constituiría una **fuga de información**: hay que elegir el objetivo y descartar el otro.

2. **`NSP` es ordinal.** Normal < Sospechoso < Patológico describe una gradación de gravedad. Clasificar como *sospechoso* un caso *patológico* es un error mucho menos grave que clasificarlo como *normal*. Conviene acompañar las métricas nominales de índices **sensibles al orden** (QWK, MAE ordinal) y examinar la matriz de confusión.

3. **El desequilibrio no es un detalle, es el problema.** Un clasificador que prediga siempre «Normal» alcanza un 78 % de exactitud sin haber aprendido nada. La métrica relevante es el **recall de la clase patológica**: en obstetricia, un falso negativo (declarar normal un feto en sufrimiento) tiene un coste incomparablemente mayor que un falso positivo.

4. **Multicolinealidad en el histograma.** Las diez variables de ese bloque comparten información. Esto no perjudica la capacidad predictiva de los modelos basados en árboles, pero sí **distorsiona la interpretación**: tanto la importancia por impureza como la de permutación repartirán arbitrariamente la relevancia entre `Mode`, `Mean` y `Median`. Los valores SHAP son aquí la herramienta adecuada.

5. **Variables casi constantes.** `DS` (deceleraciones severas) y `Nzeros` toman el valor cero en la gran mayoría de los registros. Aportan poca información, pero cuando no son nulas pueden ser **muy informativas**: conviene no descartarlas mecánicamente por su baja varianza.

## <font color="steelblue">Reglas del juego (buenas prácticas obligatorias)</font>

1. **Elige y razona la formulación:** trabaja con **`NSP`** (3 clases) como tarea principal; deja `CLASS` (10 clases) y el enfoque **jerárquico** como extensión.
2. **Evita la fuga entre objetivos:** si predices `NSP`, **quita `CLASS` de `X`** (y viceversa). Comprueba que las columnas indicadoras (`A`…`SUSP`) ya **no** están.
3. **Prioridad clínica:** el peor error es clasificar un caso **Patológico como Normal**; vigila el **recall de Patológico** (y de Sospechoso).
4. **Partición estratificada**; el *test* solo se toca al final.
5. **Sin fuga en el preprocesado:** escalado/remuestreo dentro de un **`Pipeline`**.
6. **Equilibrado solo en *train*** (material 11): `NSP` está **desequilibrado**.
7. **Reproducibilidad y honestidad:** `random_state` fijado; reporta lo que no funcionó.

# <font color="steelblue">Fase 0 — Preparación del entorno y carga de datos</font>

In [ ]:
# !pip -q install kagglehub imbalanced-learn gradio optuna scikit-learn shap
import os, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings('ignore'); sns.set_theme(style='whitegrid')
import kagglehub
RNG = 42

In [ ]:
# Descarga y carga (equivalente a usar %cd $path y leer el CSV)
path = kagglehub.dataset_download("akshat0007/fetalhr")
print("Ruta:", path, "| Archivos:", os.listdir(path))
cardiogr = pd.read_csv(os.path.join(path, "CTG.csv"))

# Quitamos metadatos, la línea de base redundante (LBE) y las columnas indicadoras
# A..SUSP, que codifican los propios objetivos (su uso sería FUGA). Conservamos CLASS y NSP.
cardiogr = cardiogr.drop(columns=['FileName', 'Date', 'SegFile', 'b', 'e', 'LBE',
                                  'A', 'B', 'C', 'D', 'E', 'AD', 'DE', 'LD', 'FS', 'SUSP'])
print(f"Dimensiones: {cardiogr.shape[0]:,} filas × {cardiogr.shape[1]} columnas")
cardiogr.head()

# <font color="steelblue">Fase 1 — Comprensión, EDA y elección de la formulación</font>

**Tareas obligatorias**
1. **Dos objetivos.** Confirmad que existen `NSP` (3 clases) y `CLASS` (10 clases) y su **relación jerárquica** (patrones → estados). Justificad por qué centráis el proyecto en **`NSP`**.
2. **Distribución de `NSP`.** Calculadla: veréis fuerte **desequilibrio** (Normal ≫ Sospechoso > Patológico). Identificad **Patológico** como la **clase crítica**.
3. **Variables.** Confirmad que son todas numéricas y revisad rangos (p. ej. `LB` 110–160 lpm normal). **Multicolinealidad:** matriz de correlación de las variables del histograma.
4. **Relación con el objetivo.** ¿Qué señales se asocian a Patológico (`ASTV`/`ALTV` altas, `DP`/`DS`, `MSTV` baja…)? Boxplots por estado.
5. **Conclusión:** 3–4 hallazgos.

> **A responder:** con este desequilibrio y una clase crítica, ¿qué métricas usaréis? (pista: **macro-F1**, exactitud balanceada y, sobre todo, **recall de Patológico**).

# <font color="steelblue">Fase 2 — Preprocesado y partición</font>

**Tareas obligatorias**
1. **Objetivo:** `y = NSP` (mapead a 0/1/2 si queréis, guardando el inverso: 0=Normal, 1=Sospechoso, 2=Patológico).
2. **Anti-fuga:** `X` = todas las características **quitando AMBOS objetivos** (`NSP` **y** `CLASS`). Verificad que `A`…`SUSP` no están.
3. **Escalado:** todas las variables son numéricas; **escalad** para logística/SVM/kNN (los árboles no lo requieren), dentro de un **`Pipeline`**.
4. **(Opcional) Multicolinealidad:** valorad **selección de variables** o **PCA** para los modelos lineales, y comparad.
5. **Partición estratificada**.

> **A responder:** ¿por qué dejar `CLASS` en `X` sería una fuga al predecir `NSP`?

# <font color="steelblue">Fase 3 — Modelos base y comparación</font>

**Tareas obligatorias**
1. Comparad **≥5 familias** del curso: **Regresión logística (multinomial)**, **kNN**, **SVM**, **Árbol**, **Random Forest**, **HistGradientBoosting** (o XGBoost/LightGBM/CatBoost), **Naive Bayes**.
2. **Validación cruzada repetida** estratificada con una métrica adecuada al desequilibrio (**`f1_macro`** o **`balanced_accuracy`**).
3. **Tabla** comparativa y comentario; mirad también el **recall por clase** (no solo el promedio).

# <font color="steelblue">Fase 4 — Ponderación de muestras y equilibrado</font>

`NSP` está **claramente desequilibrado** y la clase crítica (**Patológico**) es minoritaria: el equilibrado **importa**. Comparad, sobre los 2–3 mejores modelos:

1. **Sin tratamiento** (línea base).
2. **Sensible al coste:** `class_weight='balanced'`.
3. **Sobremuestreo:** **SMOTE** (multiclase; dentro del `ImbPipeline`).
4. (Opcional) submuestreo/híbrido.

Reportad **macro-F1**, **recall de Patológico** y **de Sospechoso**, y exactitud balanceada. Discutid el compromiso: subir el *recall* de Patológico suele bajar la precisión global.

> **Sin fugas:** remuestreo dentro de `Pipeline` de *imbalanced-learn*, solo en *train*.

# <font color="steelblue">Fase 5 — Optimización de hiperparámetros</font>

1. Optimizad los **2–3 mejores** (modelo + equilibrado).
2. `GridSearchCV`/`RandomizedSearchCV`/**Optuna**, con CV estratificada y la métrica elegida (p. ej. `f1_macro` o un *scorer* centrado en el **recall de Patológico**); búsqueda **sobre el `Pipeline`** (prefijo `clf__`).
3. (Recomendado) **CV anidada**.
4. Reportad mejores hiperparámetros y la mejora.

# <font color="steelblue">Fase 6 — Combinación de modelos</font>

1. Combinad los mejores con **`VotingClassifier`** (votación **blanda**) y/o **`StackingClassifier`**.
2. Comparad frente al **mejor individual** (macro-F1 / recall de Patológico): ¿mejora?
3. **Combinad solo si aporta** mejora real (requisito: *si fuera necesario*).

# <font color="steelblue">Fase 7 — Evaluación, coste del error e interpretación</font>

El *test* se usa una sola vez.

**Tareas obligatorias**
1. **Métricas finales:** **matriz de confusión** 3×3, `classification_report`, **macro-F1**, **recall por clase** y exactitud balanceada.
2. **Coste del error (clave).** Analizad las celdas **Patológico → Normal** (el error más peligroso). ¿Qué política de umbral/coste reduce esos infra-diagnósticos? Comparad.
3. **Interpretabilidad (SHAP).** ¿Pesan `ASTV`, `ALTV`, `MSTV`, `DP`/`DS` como indica la clínica del bienestar fetal?
4. **Extensión distintiva (recomendada):**
   * **10 clases:** repetid el flujo con `CLASS` (quitando `NSP` de `X`) y comparad la dificultad.
   * **Jerárquico:** predecid `CLASS` y **agregad** a `NSP` (mapeo patrón→estado); comparad con el modelo directo de `NSP`. ¿Ayuda la jerarquía?
5. **Discusión crítica:** subjetividad de la etiqueta de consenso, variabilidad inter-observador, papel del CAD como **apoyo**.

# <font color="steelblue">Fase 8 — Despliegue del modelo</font>

1. **Persistencia:** guardad el **`Pipeline` completo** con `joblib`.
2. **Función de predicción:** `predecir_estado(...)` con las características CTG que devuelva el **estado** (`Normal/Sospechoso/Patológico`, con el mapeo inverso) y las **probabilidades**.
3. **Interfaz interactiva:** app con **Gradio** (o `ipywidgets`) con campos para las características principales (`LB`, `ASTV`, `ALTV`, `MSTV`, `DP`, `UC`…); salida = estado fetal. En Colab da un **enlace público** (incluidlo).
4. (Opcional, nota extra) **Streamlit**/**FastAPI**.

> **Aviso clínico (obligatorio en la interfaz):** herramienta **educativa** de apoyo; **no** sustituye la interpretación del CTG por un profesional.

# <font color="steelblue">Pistas y errores típicos</font>

* **Fuga entre objetivos.** Al predecir `NSP`, quita **`CLASS`** de `X` (y confirma que `A`…`SUSP` ya no están): son etiquetas, no predictores.
* **La clase crítica manda.** En bienestar fetal, un **Patológico** clasificado como **Normal** es el peor error: prioriza su **recall** y míralo por clase, no solo el promedio.
* **Multicolinealidad del histograma.** `Mode`/`Mean`/`Median`/`Min`/`Max`/`Width` están muy correlacionadas: afecta a modelos lineales (considera selección/PCA); los árboles lo toleran.
* **Dos formulaciones.** `NSP` (3) es la clínica; `CLASS` (10) es de grano fino. La **jerarquía** patrón→estado es una extensión muy informativa.
* **Despliegue:** guarda el **Pipeline entero** y respeta el orden/formato de las columnas.

# <font color="steelblue">Referencias</font>

* Ayres-de-Campos, D. et al. (2000). *SisPorto 2.0: automated analysis of cardiotocograms*. J. Maternal-Fetal Medicine, 9(5).
* Campos, D. & Bernardes, J. (2000). *Cardiotocography*. UCI ML Repository.
* *Cardiotocography Data Analysis to Predict Fetal Health Risks with Tree-Based Ensemble Learning*. IJITCS, 2021.
* Grivell, R. M. et al. (2015). *Antenatal cardiotocography for fetal assessment*. Cochrane Database Syst. Rev.
* Cuadernos del curso: *Equilibrando las muestras*, *Random Forest*, *Boosting*, *SVM*, *Regresión logística múltiple*.